In [ ]:
# Email Reply Draft Assistant

## 1 — Project Overview

This notebook builds an **email reply draft assistant** that reads an incoming email and:

1. **Classifies** its intent (complaint, inquiry, meeting request, follow-up, etc.)
2. **Determines** urgency and suggested reply tone
3. **Generates 3 reply drafts** in different lengths (concise / balanced / detailed)
4. **Supports tone control** — switch between formal, friendly, empathetic styles

**Key engineering patterns taught here:**
- **Structured output** — using Pydantic schemas to force LLM output into a predictable shape
- **Two-stage pipeline** — classify first, then use the classification to guide generation
- **Prompt template design** — how to write prompts that produce reliable, formatted output

**Stack:**
| Component | Tool | Why |
|-----------|------|-----|
| Orchestration | LangChain (LCEL) | Clean chain composition with structured output |
| Schema validation | Pydantic | Type-safe output parsing |
| LLM | Ollama (local) | Free, no API keys, runs offline |

**Prerequisite:** [Ollama](https://ollama.com/) must be installed and running with a model pulled (e.g., `ollama pull qwen3:8b`).

## 2 — Learning Goals

By the end of this notebook you will be able to:

1. **Use Pydantic models with LLM output** — define schemas that the LLM must follow, with typed fields and descriptions
2. **Build a two-stage LLM pipeline** — classification → generation, where stage 1 output feeds stage 2
3. **Design effective prompt templates** — understand why structure, role, examples, and format instructions matter
4. **Implement tone control** — generate multiple stylistic variations from the same input
5. **Compare naive single-prompt vs. structured multi-step approaches** — understand quality and reliability tradeoffs
6. **Handle structured output parsing failures** — what to do when the LLM doesn't follow your schema

## 3 — Problem Statement

**Problem:** You receive dozens of emails daily — complaints, inquiries, meeting requests, follow-ups. Each needs a different tone and level of detail. Writing replies is time-consuming and repetitive.

**What we want:**
- Given an incoming email, automatically classify what it's about
- Suggest the right tone (formal for executives, empathetic for complaints, friendly for colleagues)
- Generate 3 reply drafts at different lengths so the user can pick and edit

**Why two stages?**
A single prompt like "read this email and write a reply" works for simple cases but fails when:
- You need consistent structure (subject + body + tone label)
- You want the classification to be reusable (e.g., for routing, prioritization)
- You need multiple output variations

**Our approach:** Classify first → use classification metadata to guide reply generation.

In [ ]:
## 4 — Why This Project Matters

**Email automation is one of the highest-ROI GenAI applications in enterprise:**
- Gmail's Smart Reply / Smart Compose serves billions of suggestions daily
- Salesforce Einstein, HubSpot, and Zendesk all use LLMs for email drafting
- Customer support teams use intent classification for ticket routing before any reply is generated

**The engineering patterns here transfer directly to production:**

| Pattern | Used in this project | Also used in |
|---------|---------------------|-------------|
| Structured output (Pydantic) | Email classification | API backends, data extraction, form filling |
| Two-stage pipeline | Classify → Generate | Triage → Route, Analyze → Act |
| Tone control | Formal/friendly/empathetic | Brand voice, audience targeting |
| Multi-option generation | 3 reply drafts | A/B testing, human-in-the-loop review |

**Why structured output matters so much:**
Without it, you get free-form text that you have to regex-parse, which is brittle. Pydantic + JsonOutputParser gives you typed, validated data every time.

## 5 — Dataset Overview

We use **manually crafted sample emails** representing common business email categories. This is intentional — in a GenAI prompt-engineering project, the "data" is the input text, and using hand-crafted examples lets us control for quality and variety.

**Email categories covered:**
| Intent | Example scenario | Expected tone |
|--------|-----------------|---------------|
| Complaint | Defective product, poor service | Empathetic |
| Inquiry | Pricing question, feature request | Professional |
| Meeting request | Schedule alignment | Friendly |
| Follow-up | Checking status of a prior request | Professional |
| Feedback | Positive or negative product review | Appreciative |

**Why not use a real email dataset?**
Real email datasets (like Enron) contain personal/sensitive information and are pre-labeled for different tasks (fraud detection, not intent classification). For prompt engineering, curated examples with known ground truth are more instructive.

In [ ]:
## 6 — Environment Setup

**Required:** Ollama installed and running locally with a model available.

No API keys needed — everything runs locally.

# Install if needed (uncomment)
# !pip install langchain langchain-ollama -q

import shutil

if shutil.which("ollama") is None:
    print("WARNING: Ollama not found. Install from https://ollama.com/")
    print("Then run: ollama pull qwen3:8b")
else:
    print("Ollama found.")

print("Setup complete.")

In [ ]:
## 7 — Imports

import json
import textwrap
from pathlib import Path

from langchain_ollama import ChatOllama
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from pydantic import BaseModel, Field

print("All imports successful!")

In [ ]:
## 8 — Data Loading (Sample Emails)

We define a set of realistic sample emails across different intent categories. Each has a known ground-truth intent so we can evaluate classification accuracy later.

SAMPLE_EMAILS = {
    "complaint": {
        "expected_intent": "complaint",
        "expected_urgency": "high",
        "text": """Subject: Extremely disappointed with recent order #4582

Hi,

I ordered the Premium Wireless Headphones (Order #4582) two weeks ago and they
arrived with a cracked case and the left ear cup doesn't produce any sound.
This is the second defective product I've received from your store.

I need an immediate replacement or a full refund. I've been a loyal customer
for 3 years and this experience has been really frustrating.

Please respond ASAP.

Thanks,
Sarah Johnson""",
    },
    "meeting_request": {
        "expected_intent": "meeting_request",
        "expected_urgency": "medium",
        "text": """Subject: Q3 Planning Session - Can we meet Thursday?

Hi team,

I'd like to schedule a Q3 planning session to discuss our roadmap priorities
and resource allocation. Would Thursday at 2 PM work for everyone?

Agenda:
1. Review Q2 metrics
2. Discuss Q3 OKRs
3. Resource planning

Please confirm your availability.

Best,
Mike Chen""",
    },
    "inquiry": {
        "expected_intent": "inquiry",
        "expected_urgency": "medium",
        "text": """Subject: Question about Enterprise API pricing

Hello,

We're evaluating your API platform for our company (500+ employees).
Could you provide information on:
- Enterprise pricing tiers
- SLA guarantees
- Custom integration support

We'd also like to know if you offer a pilot program.

Regards,
David Park
CTO, TechCorp""",
    },
    "follow_up": {
        "expected_intent": "follow_up",
        "expected_urgency": "medium",
        "text": """Subject: Re: Project proposal status?

Hi Lisa,

Just following up on the project proposal I sent last Tuesday. Have you
had a chance to review it with the team?

Happy to answer any questions or hop on a quick call if that helps.

Thanks,
James""",
    },
    "feedback": {
        "expected_intent": "feedback",
        "expected_urgency": "low",
        "text": """Subject: Great experience with your support team

Hi,

I wanted to share that your support team did an excellent job resolving
my billing issue last week. Alex was patient, knowledgeable, and resolved
everything in under 10 minutes.

This kind of service is why I keep coming back. Keep it up!

Best regards,
Maria Santos""",
    },
}

print(f"Loaded {len(SAMPLE_EMAILS)} sample emails:")
for key, data in SAMPLE_EMAILS.items():
    word_count = len(data["text"].split())
    print(f"  {key}: {word_count} words, expected intent={data['expected_intent']}")

In [ ]:
## 9 — Data Inspection

Let's look at the structure and characteristics of our sample emails to understand what the LLM will be working with.

In [ ]:
# Inspect each email's structure
for key, data in SAMPLE_EMAILS.items():
    text = data["text"]
    lines = text.strip().split("\n")
    subject = next((l for l in lines if l.startswith("Subject:")), "No subject")
    has_greeting = any(l.strip().startswith(("Hi", "Hello", "Dear")) for l in lines)
    has_closing = any(l.strip().startswith(("Thanks", "Best", "Regards")) for l in lines)

    print(f"--- {key.upper()} ---")
    print(f"  Subject: {subject.replace('Subject: ', '')}")
    print(f"  Lines: {len(lines)}, Words: {len(text.split())}")
    print(f"  Has greeting: {has_greeting}, Has closing: {has_closing}")
    print(f"  Expected: intent={data['expected_intent']}, urgency={data['expected_urgency']}")
    print()

## Step 6 — Build the Reply Generator

Using the classification results, we generate **3 reply options**:
1. **Concise** — short and to the point
2. **Balanced** — professional with adequate detail
3. **Detailed** — thorough and comprehensive

In [ ]:
reply_parser = JsonOutputParser(pydantic_object=ReplyOptions)

reply_prompt = ChatPromptTemplate.from_template(
    """You are a professional email writer. Based on the email classification below,
generate exactly 3 reply drafts.

Email Classification:
- Intent: {intent}
- Urgency: {urgency}
- Suggested Tone: {tone}
- Key Points to Address: {key_points}

Original Email:
---
{email}
---

Generate 3 replies:
1. CONCISE - Brief, 2-3 sentences
2. BALANCED - Professional, medium length
3. DETAILED - Thorough, addresses every point

All replies should match the suggested tone ({tone}).

{format_instructions}

Provide the 3 replies as JSON:"""
)

reply_chain = reply_prompt | llm | reply_parser

print("Reply generator chain ready!")

In [ ]:
# Generate replies for the complaint email
key_points_str = ", ".join(classification.get("key_points", []))

replies = reply_chain.invoke({
    "intent": classification.get("intent", "complaint"),
    "urgency": classification.get("urgency", "high"),
    "tone": classification.get("suggested_tone", "empathetic"),
    "key_points": key_points_str,
    "email": email_text,
    "format_instructions": reply_parser.get_format_instructions(),
})

# Display the replies
for reply in replies.get("replies", []):
    print(f"\n{'='*60}")
    print(f"📝 Style: {reply['style'].upper()}")
    print(f"📋 Subject: {reply['subject']}")
    print(f"{'-'*60}")
    print(reply['body'])

## Step 7 — Complete Pipeline Function

Let's wrap everything into a single function for easy use.

In [ ]:
def process_email(email_text: str) -> dict:
    """Complete pipeline: classify email → generate 3 reply options."""
    
    # Step 1: Classify
    print("🔍 Classifying email...")
    classification = classify_chain.invoke({
        "email": email_text,
        "format_instructions": classification_parser.get_format_instructions(),
    })
    
    print(f"  Intent: {classification.get('intent')}")
    print(f"  Urgency: {classification.get('urgency')}")
    print(f"  Tone: {classification.get('suggested_tone')}")
    print(f"  Summary: {classification.get('summary')}")
    
    # Step 2: Generate replies
    print("\n✍️ Generating reply drafts...")
    key_points_str = ", ".join(classification.get("key_points", []))
    
    replies = reply_chain.invoke({
        "intent": classification.get("intent", "unknown"),
        "urgency": classification.get("urgency", "medium"),
        "tone": classification.get("suggested_tone", "professional"),
        "key_points": key_points_str,
        "email": email_text,
        "format_instructions": reply_parser.get_format_instructions(),
    })
    
    # Display
    for i, reply in enumerate(replies.get("replies", []), 1):
        print(f"\n{'='*60}")
        print(f"Option {i}: {reply['style'].upper()}")
        print(f"Subject: {reply['subject']}")
        print(f"{'-'*60}")
        print(reply['body'])
    
    return {"classification": classification, "replies": replies}


# Try with the inquiry email
result = process_email(sample_emails["inquiry"])

## 🧠 Key Concepts Recap

| Concept | What it does |
|---------|-------------|
| **Pydantic Models** | Define exact output schema with types and descriptions |
| **JsonOutputParser** | Forces LLM output to match our Pydantic schema |
| **Format Instructions** | Auto-generated text telling the LLM how to format output |
| **Chain Composition** | `prompt \| llm \| parser` — LCEL for clean pipelines |
| **Two-Stage Pipeline** | Classify first, then use classification to guide generation |

## 🔧 Customization Ideas

- **Add more intents**: "job application", "newsletter", "legal notice"
- **Tone presets**: Let users override the suggested tone
- **Context injection**: Add company name, sender role, or previous thread
- **Multi-language**: Add `reply_language` field to generate replies in other languages